In [1]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [5]:
def preprocess_mask(mask):
    # Expected original BraTS shape
    if mask.shape != (240, 240, 155):
        raise ValueError(f"Unexpected mask shape: {mask.shape}")

    # Apply exactly the same spatial crop as T2f
    mask = mask[16:224, 8:232, :]

    # Apply exactly the same z-padding as T2f
    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(np.int64)

In [6]:
def calculate_tumour_entropy(image, mask, num_bins=256):
    # Whole tumour = all non-zero tumour labels
    tumour_region = mask > 0

    if not np.any(tumour_region):
        raise ValueError("No tumour voxels found")

    tumour_values = image[tumour_region]

    # T2f has already been normalized to [0, 1]
    tumour_values = np.clip(tumour_values, 0.0, 1.0)

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = hist.astype(np.float64)
    probabilities = probabilities / probabilities.sum()

    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return np.float32(entropy)

In [7]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]
        seg_file = [f for f in files if "seg" in f.lower()][0]

        # Load T2f
        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        # Load segmentation
        mask = nib.load(
            os.path.join(subject_path, seg_file)
        ).get_fdata()

        # Apply preprocessing
        image = preprocess_t2f(image)
        mask = preprocess_mask(mask)

        # Calculate whole-tumour Shannon entropy
        entropy = calculate_tumour_entropy(
            image,
            mask
        )

        # Convert to tensors
        image = torch.from_numpy(
            image
        ).float().unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).long().unsqueeze(0)

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [8]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [10]:
# ============================================================
# Conditional LDM V7
# 3D + 2D building blocks for Triplane VAE
# ============================================================


class VAEBlock3D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),

            nn.SiLU(),

            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),

            nn.SiLU()
        )


        self.skip = (

            nn.Identity()

            if in_channels == out_channels

            else nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        )


    def forward(self, x):

        return (
            self.block(x)
            +
            self.skip(x)
        )


class VAEBlock2D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),

            nn.SiLU(),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),

            nn.SiLU()
        )


        self.skip = (

            nn.Identity()

            if in_channels == out_channels

            else nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        )


    def forward(self, x):

        return (
            self.block(x)
            +
            self.skip(x)
        )


In [11]:
# ============================================================
# Conditional LDM V7
# Learned Triplane Encoder
#
# MRI:
# [B, 1, 208, 224, 160]
#
# 3D feature volume:
# [B, 128, 52, 56, 40]
#
# Triplanes:
# XY = [B, 32, 52, 56]
# XZ = [B, 32, 52, 40]
# YZ = [B, 32, 56, 40]
# ============================================================


class TriplaneEncoder3D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        base_channels=32,
        plane_channels=32
    ):
        super().__init__()


        # ====================================================
        # Shared 3D encoder
        # ====================================================

        self.enc1 = VAEBlock3D(
            in_channels,
            base_channels
        )


        self.down1 = nn.Conv3d(
            base_channels,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )


        self.enc2 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )


        self.down2 = nn.Conv3d(
            base_channels * 2,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1
        )


        self.bottleneck = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )


        feature_channels = (
            base_channels * 4
        )


        # ====================================================
        # Learned plane projections
        #
        # Input to each projection:
        # [B, 128, spatial1, spatial2]
        #
        # Output:
        # [B, 32, spatial1, spatial2]
        # ====================================================

        self.xy_projection = nn.Sequential(

            nn.Conv2d(
                feature_channels,
                plane_channels,
                kernel_size=1
            ),

            VAEBlock2D(
                plane_channels,
                plane_channels
            )
        )


        self.xz_projection = nn.Sequential(

            nn.Conv2d(
                feature_channels,
                plane_channels,
                kernel_size=1
            ),

            VAEBlock2D(
                plane_channels,
                plane_channels
            )
        )


        self.yz_projection = nn.Sequential(

            nn.Conv2d(
                feature_channels,
                plane_channels,
                kernel_size=1
            ),

            VAEBlock2D(
                plane_channels,
                plane_channels
            )
        )


        # ====================================================
        # Gaussian parameters per plane
        # ====================================================

        self.xy_mu = nn.Conv2d(
            plane_channels,
            plane_channels,
            kernel_size=1
        )

        self.xy_logvar = nn.Conv2d(
            plane_channels,
            plane_channels,
            kernel_size=1
        )


        self.xz_mu = nn.Conv2d(
            plane_channels,
            plane_channels,
            kernel_size=1
        )

        self.xz_logvar = nn.Conv2d(
            plane_channels,
            plane_channels,
            kernel_size=1
        )


        self.yz_mu = nn.Conv2d(
            plane_channels,
            plane_channels,
            kernel_size=1
        )

        self.yz_logvar = nn.Conv2d(
            plane_channels,
            plane_channels,
            kernel_size=1
        )


    def forward(self, x):

        # ----------------------------------------------------
        # 3D encoding
        # ----------------------------------------------------

        x = self.enc1(x)

        x = self.down1(x)
        x = self.enc2(x)

        x = self.down2(x)
        x = self.bottleneck(x)

        # x:
        # [B, 128, 52, 56, 40]


        # ====================================================
        # Project 3D feature volume to three planes
        #
        # Important:
        # these are learned 2D feature representations,
        # not raw MRI slices.
        # ====================================================


        # XY plane:
        # aggregate depth D

        xy_feature = x.mean(
            dim=4
        )

        # [B, 128, 52, 56]

        xy_feature = (
            self.xy_projection(
                xy_feature
            )
        )


        # XZ plane:
        # aggregate Y

        xz_feature = x.mean(
            dim=3
        )

        # [B, 128, 52, 40]

        xz_feature = (
            self.xz_projection(
                xz_feature
            )
        )


        # YZ plane:
        # aggregate X

        yz_feature = x.mean(
            dim=2
        )

        # [B, 128, 56, 40]

        yz_feature = (
            self.yz_projection(
                yz_feature
            )
        )


        # ====================================================
        # Gaussian triplane distributions
        # ====================================================

        mu_xy = self.xy_mu(
            xy_feature
        )

        logvar_xy = self.xy_logvar(
            xy_feature
        )


        mu_xz = self.xz_mu(
            xz_feature
        )

        logvar_xz = self.xz_logvar(
            xz_feature
        )


        mu_yz = self.yz_mu(
            yz_feature
        )

        logvar_yz = self.yz_logvar(
            yz_feature
        )


        return (
            mu_xy,
            logvar_xy,
            mu_xz,
            logvar_xz,
            mu_yz,
            logvar_yz
        )

In [12]:
# ============================================================
# Conditional LDM V7
# Triplane Gaussian reparameterisation
# ============================================================


def reparameterize_plane(
    mu,
    logvar
):

    std = torch.exp(
        0.5 * logvar
    )

    eps = torch.randn_like(
        std
    )

    return (
        mu
        +
        eps * std
    )

In [13]:
# ============================================================
# Conditional LDM V7
# Triplane -> 3D MRI Decoder
# ============================================================


class TriplaneDecoder3D(nn.Module):

    def __init__(
        self,
        out_channels=1,
        base_channels=32,
        plane_channels=32
    ):
        super().__init__()


        fusion_channels = (
            plane_channels * 3
        )


        # ====================================================
        # Fuse broadcast triplane features
        #
        # 96 channels
        # ->
        # 128 channels
        # ====================================================

        self.fusion = nn.Sequential(

            nn.Conv3d(
                fusion_channels,
                base_channels * 4,
                kernel_size=3,
                padding=1
            ),

            VAEBlock3D(
                base_channels * 4,
                base_channels * 4
            )
        )


        # ====================================================
        # Upsample /4 -> /2
        # ====================================================

        self.up2 = nn.ConvTranspose3d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )


        self.dec1 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )


        # ====================================================
        # Upsample /2 -> full resolution
        # ====================================================

        self.up1 = nn.ConvTranspose3d(
            base_channels * 2,
            base_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )


        self.final_block = VAEBlock3D(
            base_channels,
            base_channels
        )


        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )


    def forward(
        self,
        z_xy,
        z_xz,
        z_yz
    ):

        # ====================================================
        # Plane dimensions
        # ====================================================

        B = z_xy.shape[0]

        H = z_xy.shape[2]
        W = z_xy.shape[3]

        D = z_xz.shape[3]


        # ====================================================
        # Broadcast XY
        #
        # [B,C,H,W]
        # ->
        # [B,C,H,W,D]
        # ====================================================

        xy_3d = (
            z_xy
            .unsqueeze(-1)
            .expand(
                -1,
                -1,
                H,
                W,
                D
            )
        )


        # ====================================================
        # Broadcast XZ
        #
        # [B,C,H,D]
        # ->
        # [B,C,H,W,D]
        # ====================================================

        xz_3d = (
            z_xz
            .unsqueeze(3)
            .expand(
                -1,
                -1,
                H,
                W,
                D
            )
        )


        # ====================================================
        # Broadcast YZ
        #
        # [B,C,W,D]
        # ->
        # [B,C,H,W,D]
        # ====================================================

        yz_3d = (
            z_yz
            .unsqueeze(2)
            .expand(
                -1,
                -1,
                H,
                W,
                D
            )
        )


        # ====================================================
        # Fuse all three planes
        # ====================================================

        x = torch.cat(
            [
                xy_3d,
                xz_3d,
                yz_3d
            ],
            dim=1
        )


        x = self.fusion(x)


        # /4 -> /2

        x = self.up2(x)

        x = self.dec1(x)


        # /2 -> full

        x = self.up1(x)

        x = self.final_block(x)


        x = self.output_conv(x)

        x = torch.sigmoid(x)


        return x

In [14]:
# ============================================================
# Conditional LDM V7
# Triplane VAE
# ============================================================


class TriplaneVAE3D(nn.Module):

    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=32,
        plane_channels=32
    ):
        super().__init__()


        self.encoder = (
            TriplaneEncoder3D(
                in_channels=in_channels,
                base_channels=base_channels,
                plane_channels=plane_channels
            )
        )


        self.decoder = (
            TriplaneDecoder3D(
                out_channels=out_channels,
                base_channels=base_channels,
                plane_channels=plane_channels
            )
        )


    def forward(self, x):

        (
            mu_xy,
            logvar_xy,
            mu_xz,
            logvar_xz,
            mu_yz,
            logvar_yz
        ) = self.encoder(x)


        z_xy = reparameterize_plane(
            mu_xy,
            logvar_xy
        )


        z_xz = reparameterize_plane(
            mu_xz,
            logvar_xz
        )


        z_yz = reparameterize_plane(
            mu_yz,
            logvar_yz
        )


        reconstruction = (
            self.decoder(
                z_xy,
                z_xz,
                z_yz
            )
        )


        return (
            reconstruction,

            mu_xy,
            logvar_xy,

            mu_xz,
            logvar_xz,

            mu_yz,
            logvar_yz,

            z_xy,
            z_xz,
            z_yz
        )

In [15]:
# ============================================================
# Conditional LDM V7
# Triplane VAE loss
# ============================================================


def gradient_3d_loss_v7(
    reconstruction,
    target
):

    recon_dx = (
        reconstruction[:, :, 1:, :, :]
        -
        reconstruction[:, :, :-1, :, :]
    )

    target_dx = (
        target[:, :, 1:, :, :]
        -
        target[:, :, :-1, :, :]
    )


    recon_dy = (
        reconstruction[:, :, :, 1:, :]
        -
        reconstruction[:, :, :, :-1, :]
    )

    target_dy = (
        target[:, :, :, 1:, :]
        -
        target[:, :, :, :-1, :]
    )


    recon_dz = (
        reconstruction[:, :, :, :, 1:]
        -
        reconstruction[:, :, :, :, :-1]
    )

    target_dz = (
        target[:, :, :, :, 1:]
        -
        target[:, :, :, :, :-1]
    )


    return (
        F.l1_loss(
            recon_dx,
            target_dx
        )
        +
        F.l1_loss(
            recon_dy,
            target_dy
        )
        +
        F.l1_loss(
            recon_dz,
            target_dz
        )
    ) / 3.0



def kl_plane_loss(
    mu,
    logvar
):

    return -0.5 * torch.mean(
        1
        +
        logvar
        -
        mu.pow(2)
        -
        logvar.exp()
    )



def vae_loss_v7(
    reconstruction,
    target,

    mu_xy,
    logvar_xy,

    mu_xz,
    logvar_xz,

    mu_yz,
    logvar_yz,

    kl_weight=1e-6,
    multiscale_weight=0.1,
    gradient_weight=0.1
):

    # ========================================================
    # Full-resolution reconstruction
    # ========================================================

    recon_loss = F.l1_loss(
        reconstruction,
        target
    )


    # ========================================================
    # Multi-scale
    # ========================================================

    recon_half = F.avg_pool3d(
        reconstruction,
        kernel_size=2,
        stride=2
    )

    target_half = F.avg_pool3d(
        target,
        kernel_size=2,
        stride=2
    )


    recon_quarter = F.avg_pool3d(
        reconstruction,
        kernel_size=4,
        stride=4
    )

    target_quarter = F.avg_pool3d(
        target,
        kernel_size=4,
        stride=4
    )


    multiscale_loss = (

        0.5
        * F.l1_loss(
            recon_half,
            target_half
        )

        +

        0.5
        * F.l1_loss(
            recon_quarter,
            target_quarter
        )
    )


    # ========================================================
    # 3D detail
    # ========================================================

    gradient_loss = (
        gradient_3d_loss_v7(
            reconstruction,
            target
        )
    )


    # ========================================================
    # KL for all three planes
    # ========================================================

    kl_xy = kl_plane_loss(
        mu_xy,
        logvar_xy
    )

    kl_xz = kl_plane_loss(
        mu_xz,
        logvar_xz
    )

    kl_yz = kl_plane_loss(
        mu_yz,
        logvar_yz
    )


    kl_loss = (
        kl_xy
        +
        kl_xz
        +
        kl_yz
    ) / 3.0


    total_loss = (
        recon_loss
        +
        multiscale_weight
        * multiscale_loss
        +
        gradient_weight
        * gradient_loss
        +
        kl_weight
        * kl_loss
    )


    return (
        total_loss,
        recon_loss,
        kl_loss,
        multiscale_loss,
        gradient_loss
    )

In [16]:
# ============================================================
# Conditional LDM V7
# Triplane VAE Training
# ============================================================


def train_vae_v7(
    model,
    train_loader,
    epochs,
    optimizer,
    device,

    checkpoint_dir=(
        "vae_triplane_v7_checkpoints"
    ),

    kl_weight=1e-6,
    multiscale_weight=0.1,
    gradient_weight=0.1,

    start_epoch=0
):

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )


    history = []


    use_amp = (
        device.type == "cuda"
    )


    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=use_amp
    )


    for local_epoch in range(
        epochs
    ):

        current_epoch = (
            start_epoch
            +
            local_epoch
            +
            1
        )


        model.train()


        totals = {
            "loss": 0.0,
            "recon": 0.0,
            "kl": 0.0,
            "multi": 0.0,
            "grad": 0.0
        }


        for batch_idx, batch in enumerate(
            train_loader
        ):

            x = (
                batch["image"]
                .to(
                    device,
                    non_blocking=True
                )
            )


            optimizer.zero_grad(
                set_to_none=True
            )


            with torch.amp.autocast(
                device_type="cuda",
                enabled=use_amp
            ):

                outputs = model(x)


                (
                    reconstruction,

                    mu_xy,
                    logvar_xy,

                    mu_xz,
                    logvar_xz,

                    mu_yz,
                    logvar_yz,

                    z_xy,
                    z_xz,
                    z_yz
                ) = outputs


                (
                    loss,
                    recon_loss,
                    kl_loss,
                    multiscale_loss,
                    gradient_loss
                ) = vae_loss_v7(
                    reconstruction,
                    x,

                    mu_xy,
                    logvar_xy,

                    mu_xz,
                    logvar_xz,

                    mu_yz,
                    logvar_yz,

                    kl_weight=kl_weight,

                    multiscale_weight=(
                        multiscale_weight
                    ),

                    gradient_weight=(
                        gradient_weight
                    )
                )


            scaler.scale(
                loss
            ).backward()


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            scaler.step(
                optimizer
            )

            scaler.update()


            totals["loss"] += (
                loss.item()
            )

            totals["recon"] += (
                recon_loss.item()
            )

            totals["kl"] += (
                kl_loss.item()
            )

            totals["multi"] += (
                multiscale_loss.item()
            )

            totals["grad"] += (
                gradient_loss.item()
            )


            if (
                batch_idx + 1
            ) % 10 == 0:

                print(
                    f"Epoch "
                    f"{current_epoch}/"
                    f"{start_epoch + epochs} | "
                    f"Batch "
                    f"{batch_idx + 1}/"
                    f"{len(train_loader)} | "
                    f"Loss="
                    f"{loss.item():.6f} | "
                    f"Recon="
                    f"{recon_loss.item():.6f} | "
                    f"KL="
                    f"{kl_loss.item():.6f} | "
                    f"Multi="
                    f"{multiscale_loss.item():.6f} | "
                    f"Grad="
                    f"{gradient_loss.item():.6f}"
                )


        n = len(
            train_loader
        )


        averages = {
            key: value / n
            for key, value
            in totals.items()
        }


        history.append(
            [
                averages["loss"],
                averages["recon"],
                averages["kl"],
                averages["multi"],
                averages["grad"]
            ]
        )


        print()
        print(
            f"Epoch "
            f"{current_epoch} completed | "
            f"Loss="
            f"{averages['loss']:.6f} | "
            f"Recon="
            f"{averages['recon']:.6f} | "
            f"KL="
            f"{averages['kl']:.6f} | "
            f"Multi="
            f"{averages['multi']:.6f} | "
            f"Grad="
            f"{averages['grad']:.6f}"
        )
        print()


        checkpoint_path = os.path.join(
            checkpoint_dir,
            (
                f"vae_v7_epoch_"
                f"{current_epoch:03d}.pt"
            )
        )


        torch.save(
            {
                "epoch": current_epoch,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "plane_channels": 32,

                "base_channels": 32,

                "loss": averages["loss"]
            },
            checkpoint_path
        )


        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_v7_loss_history.npy"
            ),
            np.asarray(
                history,
                dtype=np.float32
            )
        )


        print(
            "Saved:",
            checkpoint_path
        )


    return history

In [15]:
def load_vae_checkpoint(
    model,
    optimizer,
    path,
    device
):
    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

    loaded_epoch = checkpoint["epoch"]

    print(
        f"Loaded VAE checkpoint from epoch {loaded_epoch}"
    )

    return loaded_epoch

In [18]:
# ============================================================
# Conditional LDM V7
# Triplane VAE Reconstruction
# ============================================================


@torch.no_grad()
def reconstruct_vae_v7(
    model,
    image,
    device
):

    model.eval()


    image = image.to(
        device
    )


    outputs = model(
        image
    )


    reconstruction = (
        outputs[0]
    )


    return reconstruction

In [16]:
# ============================================================
# Conditional LDM V7
# Initialise and train Triplane VAE
# ============================================================


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


vae = TriplaneVAE3D(
    in_channels=1,
    out_channels=1,
    base_channels=32,
    plane_channels=32
).to(device)


optimizer_vae = torch.optim.AdamW(
    vae.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)


print(
    "Device:",
    device
)


print(
    "V7 representation:",
    "Triplane"
)


print(
    "Plane channels:",
    32
)


print(
    "Expected plane shapes:"
)

print(
    "XY: [B,32,52,56]"
)

print(
    "XZ: [B,32,52,40]"
)

print(
    "YZ: [B,32,56,40]"
)


vae_history = train_vae_v7(
    model=vae,
    train_loader=train_loader,

    epochs=200,

    optimizer=optimizer_vae,

    device=device,

    checkpoint_dir=(
        "vae_triplane_v7_checkpoints"
    ),

    kl_weight=1e-6,

    multiscale_weight=0.1,

    gradient_weight=0.1,

    start_epoch=0
)


vae.eval()

for p in vae.parameters():
    p.requires_grad = False


print(
    "V7 Triplane VAE training complete."
)

Device: cuda
Loaded frozen VAE epoch: 15
VAE frozen: True


In [17]:
# ============================================================
# Conditional LDM V7
# Inspect learned Triplane representation
# ============================================================


vae.eval()


sample = train_dataset[0]


x = (
    sample["image"]
    .unsqueeze(0)
    .to(device)
)


with torch.no_grad():

    (
        mu_xy,
        logvar_xy,

        mu_xz,
        logvar_xz,

        mu_yz,
        logvar_yz
    ) = vae.encoder(x)


print(
    "XY plane:",
    mu_xy.shape
)

print(
    "XZ plane:",
    mu_xz.shape
)

print(
    "YZ plane:",
    mu_yz.shape
)


assert (
    tuple(mu_xy.shape)
    ==
    (1, 32, 52, 56)
)


assert (
    tuple(mu_xz.shape)
    ==
    (1, 32, 52, 40)
)


assert (
    tuple(mu_yz.shape)
    ==
    (1, 32, 56, 40)
)


print()
print(
    "Triplane representation: PASS"
)

Loaded cached latent statistics.
LATENT_MEAN: tensor([-0.0313, -0.1939,  0.1352,  0.0145])
LATENT_STD: tensor([0.6804, 0.9484, 1.4222, 0.2118])


In [18]:
entropy_stats_path = (
    "conditional_ldm_v4_entropy_stats.npz"
)

if os.path.exists(
    entropy_stats_path
):

    stats = np.load(
        entropy_stats_path
    )

    ENTROPY_MEAN = float(
        stats["mean"]
    )

    ENTROPY_STD = float(
        stats["std"]
    )

    print(
        "Loaded cached entropy statistics."
    )

else:

    print(
        "Computing entropy statistics..."
    )

    entropy_values = []

    for i in range(
        len(train_dataset)
    ):

        entropy_values.append(
            train_dataset[i][
                "heterogeneity"
            ].item()
        )

    entropy_values = np.asarray(
        entropy_values,
        dtype=np.float32
    )

    ENTROPY_MEAN = float(
        entropy_values.mean()
    )

    ENTROPY_STD = float(
        entropy_values.std()
    )

    np.savez(
        entropy_stats_path,
        mean=ENTROPY_MEAN,
        std=ENTROPY_STD
    )


print(
    "Entropy mean:",
    ENTROPY_MEAN
)

print(
    "Entropy std:",
    ENTROPY_STD
)

assert ENTROPY_STD > 0

Loaded cached entropy statistics.
Entropy mean: 6.870205879211426
Entropy std: 0.33203670382499695
